In [2]:
# =====================================================
# LOGISTIC REGRESSION - BACKGROUND EXPERIMENT
#
# Classes:
#   normal
#   clustering
#   agitation
#   background
#
# Other class removed
# =====================================================

import os
import joblib
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# =====================================================
# CONFIG
# =====================================================

SEED = 42
SR = 16000

MANIFEST = "/home/feliciano/dataset_manifest.csv"
GROUP_SPLIT = "/home/feliciano/group_split.csv"

BACKGROUND_FOLDER = (
    "/media/feliciano/Aux/"
    "AI_AFS_DATASET/"
    "AFS_BEHAVIOUR_DATASET/"
    "background"
)

# =====================================================
# LOAD FISH DATA
# =====================================================

fish_df = pd.read_csv(MANIFEST)

split_df = pd.read_csv(GROUP_SPLIT)

fish_df = fish_df.merge(
    split_df,
    on="parent_file_id",
    how="inner"
)

# Remove OTHER

fish_df = fish_df[
    fish_df["label"].str.lower().isin(
        [
            "normal",
            "clustering",
            "agitation"
        ]
    )
].copy()

print("\nFish distribution:")
print(
    fish_df["label"].value_counts()
)

# =====================================================
# LOAD BACKGROUND FILES
# =====================================================

background_files = [

    os.path.join(
        BACKGROUND_FOLDER,
        f
    )

    for f in os.listdir(
        BACKGROUND_FOLDER
    )

    if f.lower().endswith(".wav")

]

background_df = pd.DataFrame({

    "clip_path":
        background_files,

    "label":
        "background"

})

print("\nBackground clips:")
print(len(background_df))

# =====================================================
# BACKGROUND SPLIT
# =====================================================

bg_train, bg_test = train_test_split(

    background_df,

    test_size=0.20,

    random_state=SEED,

    shuffle=True

)

bg_train["split"] = "train"
bg_test["split"] = "test"

background_df = pd.concat(

    [
        bg_train,
        bg_test
    ],

    ignore_index=True

)

# =====================================================
# COMBINE
# =====================================================

df = pd.concat(

    [
        fish_df,
        background_df
    ],

    ignore_index=True

)

print("\nFinal classes:")
print(
    sorted(
        df["label"].unique()
    )
)

print("\nFinal distribution:")
print(
    df["label"].value_counts()
)

# =====================================================
# FEATURE EXTRACTION
# =====================================================

def extract_features(path):

    signal, sr = librosa.load(

        path,

        sr=SR,

        mono=True

    )

    rms = np.mean(

        librosa.feature.rms(
            y=signal
        )

    )

    signal_z = (

        signal - np.mean(signal)

    ) / (

        np.std(signal) + 1e-8

    )

    features = []

    # RMS
    features.append(rms)

    # ZCR
    features.append(

        np.mean(

            librosa.feature.zero_crossing_rate(
                signal_z
            )

        )

    )

    # Centroid
    features.append(

        np.mean(

            librosa.feature.spectral_centroid(
                y=signal_z,
                sr=sr
            )

        )

    )

    # Bandwidth
    features.append(

        np.mean(

            librosa.feature.spectral_bandwidth(
                y=signal_z,
                sr=sr
            )

        )

    )

    # Rolloff
    features.append(

        np.mean(

            librosa.feature.spectral_rolloff(
                y=signal_z,
                sr=sr
            )

        )

    )

    # Contrast
    contrast = librosa.feature.spectral_contrast(

        y=signal_z,

        sr=sr,

        n_bands=6

    )

    features.extend(

        np.mean(
            contrast,
            axis=1
        )

    )

    # MFCC
    mfcc = librosa.feature.mfcc(

        y=signal_z,

        sr=sr,

        n_mfcc=20

    )

    features.extend(

        np.mean(
            mfcc,
            axis=1
        )

    )

    return np.array(
        features,
        dtype=np.float32
    )

# =====================================================
# BUILD FEATURE MATRIX
# =====================================================

X = []
y = []

for idx, row in df.iterrows():

    if idx % 1000 == 0:

        print(
            f"{idx}/{len(df)}"
        )

    try:

        features = extract_features(
            row["clip_path"]
        )

        X.append(features)

        y.append(
            row["label"]
        )

    except Exception as e:

        print(
            "ERROR:",
            row["clip_path"]
        )

        print(e)

X = np.vstack(X)

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print("\nLabel classes:")
print(encoder.classes_)

print("\nSamples:")
print(len(X))

print("\nFeatures:")
print(X.shape[1])

# =====================================================
# SPLIT
# =====================================================

train_mask = (
    df["split"] == "train"
)

test_mask = (
    df["split"] == "test"
)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("\nTrain samples:")
print(len(X_train))

print("\nTest samples:")
print(len(X_test))

# =====================================================
# SCALE FEATURES
# =====================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

# =====================================================
# LOGISTIC REGRESSION
# =====================================================

model = LogisticRegression(

    solver="saga",

    max_iter=5000,

    random_state=SEED,

    n_jobs=-1

)

print(
    "\nTraining Logistic Regression..."
)

model.fit(

    X_train,

    y_train

)

# =====================================================
# PREDICT
# =====================================================

pred = model.predict(
    X_test
)

# =====================================================
# METRICS
# =====================================================

accuracy = accuracy_score(

    y_test,

    pred

)

macro_f1 = f1_score(

    y_test,

    pred,

    average="macro"

)

weighted_f1 = f1_score(

    y_test,

    pred,

    average="weighted"

)

per_class_f1 = f1_score(

    y_test,

    pred,

    average=None

)

print("\n================================")
print("RESULTS")
print("================================")

print(
    f"Accuracy    : {accuracy:.4f}"
)

print(
    f"Macro F1    : {macro_f1:.4f}"
)

print(
    f"Weighted F1 : {weighted_f1:.4f}"
)

print("\nPer-class F1")

for cls, score in zip(

    encoder.classes_,

    per_class_f1

):

    print(
        f"{cls}: {score:.4f}"
    )

print(
    "\nClassification Report\n"
)

print(

    classification_report(

        y_test,

        pred,

        target_names=encoder.classes_

    )

)

# =====================================================
# PREDICTIONS
# =====================================================

predictions = pd.DataFrame({

    "clip_path":
        df.loc[
            test_mask,
            "clip_path"
        ].values,

    "true_label":
        encoder.inverse_transform(
            y_test
        ),

    "predicted_label":
        encoder.inverse_transform(
            pred
        )

})

predictions.to_csv(

    "lr_background_predictions.csv",

    index=False

)

# =====================================================
# CONFUSION MATRIX
# =====================================================

cm = confusion_matrix(

    y_test,

    pred

)

cm_df = pd.DataFrame(

    cm,

    index=encoder.classes_,

    columns=encoder.classes_

)

cm_df.to_csv(

    "lr_background_confusion_matrix.csv"

)

print("\nConfusion Matrix:\n")
print(cm_df)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

feature_names = [

    "rms",
    "zcr",
    "centroid",
    "bandwidth",
    "rolloff"

]

for i in range(7):

    feature_names.append(
        f"contrast_{i+1}"
    )

for i in range(20):

    feature_names.append(
        f"mfcc_{i+1}"
    )

importance = pd.DataFrame({

    "feature":
        feature_names,

    "importance":
        np.mean(
            np.abs(
                model.coef_
            ),
            axis=0
        )

})

importance = importance.sort_values(

    "importance",

    ascending=False

)

importance.to_csv(

    "lr_background_feature_importance.csv",

    index=False

)

print("\nTop Features:\n")
print(
    importance.head(15)
)

# =====================================================
# SAVE F1
# =====================================================

pd.DataFrame({

    "class":
        encoder.classes_,

    "f1":
        per_class_f1

}).to_csv(

    "lr_background_per_class_f1.csv",

    index=False

)

# =====================================================
# SAVE MODEL
# =====================================================

joblib.dump(

    model,

    "lr_background_model.pkl"

)

joblib.dump(

    encoder,

    "lr_background_encoder.pkl"

)

joblib.dump(

    scaler,

    "lr_background_scaler.pkl"

)

print("\nSaved Files:")

print("lr_background_model.pkl")
print("lr_background_encoder.pkl")
print("lr_background_scaler.pkl")
print("lr_background_predictions.csv")
print("lr_background_confusion_matrix.csv")
print("lr_background_feature_importance.csv")
print("lr_background_per_class_f1.csv")


Fish distribution:
label
normal        11349
clustering     5700
agitation      3150
Name: count, dtype: int64

Background clips:
1500

Final classes:
['agitation', 'background', 'clustering', 'normal']

Final distribution:
label
normal        11349
clustering     5700
agitation      3150
background     1500
Name: count, dtype: int64
0/21699
1000/21699
2000/21699
3000/21699
4000/21699
5000/21699
6000/21699
7000/21699
8000/21699
9000/21699
10000/21699
11000/21699
12000/21699
13000/21699
14000/21699
15000/21699
16000/21699
17000/21699
18000/21699
19000/21699
20000/21699
21000/21699

Label classes:
['agitation' 'background' 'clustering' 'normal']

Samples:
21699

Features:
32

Train samples:
17199

Test samples:
4500

Training Logistic Regression...


/home/feliciano/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



RESULTS
Accuracy    : 0.7753
Macro F1    : 0.6659
Weighted F1 : 0.7645

Per-class F1
agitation: 0.1808
background: 0.9983
clustering: 0.6253
normal: 0.8593

Classification Report

              precision    recall  f1-score   support

   agitation       0.21      0.16      0.18       390
  background       1.00      1.00      1.00       300
  clustering       0.70      0.56      0.63       870
      normal       0.82      0.90      0.86      2940

    accuracy                           0.78      4500
   macro avg       0.68      0.65      0.67      4500
weighted avg       0.76      0.78      0.76      4500


Confusion Matrix:

            agitation  background  clustering  normal
agitation          63           0           9     318
background          0         299           1       0
clustering        137           0         489     244
normal            107           0         195    2638

Top Features:

       feature  importance
13      mfcc_2    2.071317
2     centroid    1.1811